In [5]:
import requests
import json
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

url = "https://api.deepseek.com/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}
data = {
    "model": "deepseek-chat",
    "messages": [
        {"role": "system", "content": "资深python程序员"},
        {"role": "user", "content": "用一句话介绍什么是向量数据库"}
    ],
    "temperature": 0.5,
    "max_tokens": 100
}

response = requests.post(url, headers=headers, json=data)
print("状态码：", response.status_code)

if response.status_code == 200:
    result = response.json()
    print("ai回答：", result["choices"][0]["message"]["content"])
    print("token用量：", response.json()["usage"]["total_tokens"])
else:
    print("错误：", response.text)

状态码： 200
ai回答： 向量数据库是一种专门用于存储和查询高维向量数据的数据库，它通过计算向量间的相似度来实现高效的相似性搜索。
token用量： 40


In [8]:
def deepseek_chat(messages, model="deepseek-chat", temperature=0.5, max_tokens=500):
    """
    调用DeepSeek API的通用函数
    
    参数：
        messages: list，消息列表，格式为[{"role": "user", "content": "..."}]
        model: str，模型名称
        temperature: float，随机性
        max_tokens: int，最大输出长度
    
    返回：，
        str: AI的回复内容
    """
    import requests
    import os
    from dotenv import load_dotenv
    
    load_dotenv()
    api_key = os.getenv("DEEPSEEK_API_KEY")
    
    url = "https://api.deepseek.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    data = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens
    }
    
    response = requests.post(url, headers=headers, json=data)
    print("状态码：", response.status_code)
    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"]
    else:
        raise Exception(f"API请求失败，状态码：{response.status_

In [9]:
messages = [
    {"role": "system", "content": "你是一个幽默的助手。"},
    {"role": "user", "content": "讲个冷笑话。"}
]
reply = deepseek_chat(messages, temperature=0.8)
print(reply)

状态码： 200
为什么数学书总是很忧郁？  
因为它有太多“问题”要解决，而且每次翻开都发现“答案”还在后面几页……（翻页声：哗啦——）


In [11]:
import time

def deepseek_chat_with_retry(messages, max_retries=3, **kwargs):
    for attempt in range(max_retries):
        try:
            return deepseek_chat(messages, **kwargs)
        except Exception as e:
            print(f"第{attempt+1}次尝试失败：{e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # 指数退避
            else:
                raise e

In [12]:
messages = [
    {"role": "system", "content": "你是一个幽默的助手。"},
    {"role": "user", "content": "讲个冷笑话。"}
]
reply = deepseek_chat(messages, temperature=0.8)
print(reply)

状态码： 200
为什么数学书总是很忧郁？  
因为它有太多“问题”要解决，而且每次翻开都是“新章节”的烦恼……（说完后默默裹紧外套）


In [1]:
from langchain_openai import OpenAIEmbeddings  # 注意：需用OpenAI兼容接口指向DeepSeek
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
import os


# 配置DeepSeek的嵌入模型（与OpenAI兼容）
embeddings = OpenAIEmbeddings(
    model="text-embedding-v2",  # DeepSeek的嵌入模型名
    openai_api_key=os.getenv("DEEPSEEK_API_KEY"),
    openai_api_base="https://api.deepseek.com/v1"
)

# 加载之前准备的文档（以notice.txt为例）
loader = TextLoader("./data/notice.txt", encoding="utf-8")
docs = loader.load()

# 切分文档
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = text_splitter.split_documents(docs)
print(f"文档块数量：{len(chunks)}")

# 为每个块生成向量并存入Chroma（首次运行会下载模型，稍慢）
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"  # 向量数据库持久化目录
)
print("向量数据库创建完成！")

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
import os

In [7]:
# 配置DeepSeek的嵌入模型（与OpenAI兼容）
embeddings = OpenAIEmbeddings(
    model="text-embedding-v3",  # DeepSeek的嵌入模型名
    openai_api_key=os.getenv("DEEPSEEK_API_KEY"),
    openai_api_base="https://api.deepseek.com/v1"
)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [8]:
from langchain_openai import OpenAIEmbeddings
import os
from dotenv import load_dotenv

load_dotenv()
# 配置 SiliconFlow 的嵌入模型
embeddings = OpenAIEmbeddings(
    model="BAAI/bge-large-zh-v1.5",  # 硅基流动支持的 BGE 中文模型
    openai_api_key=os.getenv("SiliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1"
)

# 测试生成向量
vector = embeddings.embed_query("这是一个测试句子")
print("向量维度:", len(vector))

向量维度: 1024


In [1]:
import os
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv

# 加载环境变量（包含硅基流动的 API Key）
load_dotenv()

# 1. 使用你已经配好的嵌入模型（硅基流动）
embeddings = OpenAIEmbeddings(
    model="BAAI/bge-large-zh-v1.5",
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1"
)



# 2. 加载你的文档（假设已有 data/notice.txt）
#这里改文档
loader = TextLoader("./data/library.txt", encoding="utf-8")
documents = loader.load()
print(f"加载了 {len(documents)} 个文档")

# 3. 切分文档
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # 每个块的大小
    chunk_overlap=50,    # 块之间重叠
    separators=["\n\n", "\n", "。", "！", "？", "，", " ", ""]
)
chunks = text_splitter.split_documents(documents)
print(f"文档被切分为 {len(chunks)} 个块")

# 4. 创建向量数据库（首次运行会生成向量并持久化）
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"   # 持久化目录
)
print("向量数据库创建完成！")

""" 5. 测试检索：查找与问题最相似的文档块
query = "方闻馆的咨询台和阅览室开放时间？"
retrieved_docs = vectorstore.similarity_search(query, k=2)
print(f"\n检索到的相关块（共 {len(retrieved_docs)} 个）：")
for i, doc in enumerate(retrieved_docs):
    print(f"块{i+1}: {doc.page_content[:300]}...")
"""
# 6. 定义 RAG 问答函数（结合检索结果 + 大模型生成）
from langchain_openai import ChatOpenAI

# 使用硅基流动的聊天模型（也可以用 DeepSeek，但为保持一致，这里也用硅基流动）
llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3",   # 硅基流动支持的模型，具体名称请查文档
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.3
)

def rag_answer(question, vectorstore, k=2):
    # 检索
    docs = vectorstore.similarity_search(question, k=k)
    context = "\n\n".join([doc.page_content for doc in docs])
    # 构造提示词
    prompt = f"""你是一个校园助手。请根据以下参考资料思考并回答用户的问题。
如果参考资料中确实没有相关信息，请如实说“我不知道”。

参考资料：
{context}

问题：{question}
回答："""
    response = llm.invoke(prompt)
    return response.content

# 测试 RAG 问答
answer = rag_answer("紫金港主馆周一至周日开馆时间是几点", vectorstore)
print("\n最终答案：", answer)



加载了 1 个文档
文档被切分为 5 个块
向量数据库创建完成！

最终答案： 根据提供的参考资料，没有提到“紫金港主馆”的相关信息。因此，我不知道紫金港主馆的开馆时间。
